# 単語に区切らず文字 n-gram で検索する (表記ゆれに強い検索)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/20_search/char_ngram_search.ipynb)

NLP クックブック 20-04 ／ 検索

形態素解析の区切り方や表記ゆれのせいで検索に当たらない。単語に区切らずに、文字 2-gram で索引を作りたい。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
import unicodedata
import numpy as np
from fugashi import Tagger
from sklearn.feature_extraction.text import TfidfVectorizer

docs = [
    "東京都立大学のサーバーにコンピューターを接続する。",
    "大阪府立図書館でインターフェイスの本を借りた。",
    "引っ越しの見積もりをウェブで申し込んだ。",
    "京都の寺を巡るツアーに参加した。",
    "新しいプリンタのドライバをインストールした。",
    "国際会議で機械翻訳の研究を発表した。",
]
queries = ["コンピュータ", "サーバ", "インターフェース", "引越し 見積り", "プリンター ドライバー", "都立大", "京都"]

tagger = Tagger()
norm = lambda s: unicodedata.normalize("NFKC", s).lower()
word_vec = TfidfVectorizer(tokenizer=lambda s: [w.surface for w in tagger(norm(s))], token_pattern=None, lowercase=False)
char_vec = TfidfVectorizer(analyzer="char", ngram_range=(2, 2), preprocessor=lambda s: norm(s).replace(" ", ""), lowercase=False)
Xw, Xc = word_vec.fit_transform(docs), char_vec.fit_transform(docs)

def top(vec, X, q):
    sims = (X @ vec.transform([q]).T).toarray().ravel()
    i = int(sims.argmax())
    return f"{sims[i]:.2f} {docs[i][:16]}" if sims[i] > 0 else "(ヒットなし)"

for q in queries:
    print(f"{q}\n   単語: {top(word_vec, Xw, q)}\n   文字 2-gram: {top(char_vec, Xc, q)}")

コンピュータ
   単語: (ヒットなし)
   文字 2-gram: 0.47 東京都立大学のサーバーにコンピュ
サーバ
   単語: (ヒットなし)
   文字 2-gram: 0.29 東京都立大学のサーバーにコンピュ
インターフェース
   単語: (ヒットなし)
   文字 2-gram: 0.44 大阪府立図書館でインターフェイス
引越し 見積り
   単語: (ヒットなし)
   文字 2-gram: 0.32 引っ越しの見積もりをウェブで申し
プリンター ドライバー
   単語: (ヒットなし)
   文字 2-gram: 0.48 新しいプリンタのドライバをインス
都立大
   単語: 0.35 東京都立大学のサーバーにコンピュ
   文字 2-gram: 0.29 東京都立大学のサーバーにコンピュ
京都
   単語: 0.38 京都の寺を巡るツアーに参加した。
   文字 2-gram: 0.23 京都の寺を巡るツアーに参加した。


In [3]:
# 文字 n-gram の弱点: 単語の一部に一致してしまう
print([g for g in char_vec.build_analyzer()("東京都")])
for q in ["京都", "都立"]:
    sims = (Xc @ char_vec.transform([q]).T).toarray().ravel()
    print(q, "→", [(round(float(s), 2), docs[i][:10]) for i, s in enumerate(sims) if s > 0])

['東京', '京都']
京都 → [(0.17, '東京都立大学のサーバ'), (0.23, '京都の寺を巡るツアー')]
都立 → [(0.21, '東京都立大学のサーバ')]


- 文字 n-gram は、長音の有無 (サーバ / サーバー)、送り仮名 (引っ越し / 引越し)、複合語の一部 (都立大) のような表記の違いに強い。検索漏れが減る。
- かわりに「東京都」の中の「京都」のような、意味のない一致が増える。単語の検索と併用し、単語で一致したものを上位にする構成が実用的。
- 日本語の全文検索エンジンでは、形態素解析による索引と 2-gram の索引を両方持つのが定石。
- 表記ゆれの吸収には、検索前の正規化 (NFKC、長音の統一、Sudachi の正規化形) も有効。